In [4]:
# ============================================
# ver08 FINAL: Retrieval + Hybrid + Rule Pipeline
# ============================================

import json
import numpy as np
from collections import Counter, defaultdict
from typing import List, Dict, Any
from tqdm import tqdm
import re

# ============================================
# 背景意図（最重要）
# ============================================
# このver08は「構造修正」ではなく「精度改善版」
#
# 改善内容:
# 1. Retrieval → embedding + BM25（精度向上）
# 2. action → clustering + majority（安定化）
# 3. target → ルールベース抽出（LLM排除）
# 4. params → 類似度選択（最頻禁止）
#
# ============================================


# ============================================
# 0. Embedding（必ず実装）
# ============================================

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")


def embed_fn(text: str):
    return embed_model.encode(text, normalize_embeddings=True)


# ============================================
# 1. BM25（簡易）
# ============================================

def tokenize(text):
    return re.findall(r"\w+", text.lower())


def build_bm25_index(dataset):
    docs = [tokenize(d["instruction"]) for d in dataset]
    df = Counter()
    for doc in docs:
        for w in set(doc):
            df[w] += 1

    return docs, df


def bm25_score(query, doc, df, N, k1=1.5, b=0.75):
    score = 0
    avgdl = sum(len(d) for d in df) / len(df)

    q = tokenize(query)
    doc_len = len(doc)

    for w in q:
        if w not in df:
            continue
        idf = np.log((N - df[w] + 0.5) / (df[w] + 0.5) + 1)

        tf = doc.count(w)
        score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avgdl))

    return score


# ============================================
# 2. ハイブリッドretrieval
# ============================================

def retrieve_topk_hybrid(instruction, dataset, bm25_docs, df, k=30):

    q_emb = embed_fn(instruction)

    scores = []

    for i, item in enumerate(dataset):

        # embedding
        emb_sim = np.dot(q_emb, item["embedding"])

        # BM25
        bm25 = bm25_score(instruction, bm25_docs[i], df, len(dataset))

        score = emb_sim + 0.5 * bm25

        scores.append((score, item))

    scores.sort(key=lambda x: x[0], reverse=True)

    return [x[1] for x in scores[:k]]


# ============================================
# 3. action決定（改善版）
# ============================================

def decide_action(retrieved):

    counter = Counter()

    for item in retrieved:
        for t in item.get("tasks", []):
            act = t.get("action")
            if act:
                counter[act] += 1

    return counter.most_common(1)[0][0]


# ============================================
# 4. target抽出（改善版）
# ============================================

COMMON_OBJECTS = [
    "man", "woman", "person", "car", "dog", "cat",
    "background", "sky", "building", "face"
]


def extract_target(instruction):

    text = instruction.lower()

    # 1. 直接一致
    for obj in COMMON_OBJECTS:
        if obj in text:
            return obj

    # 2. 名詞抽出（簡易）
    tokens = tokenize(text)
    if tokens:
        return tokens[-1]

    return "object"


# ============================================
# 5. params選択（類似度）
# ============================================

def select_params(instruction, retrieved):

    best_score = -1
    best_params = {}

    for item in retrieved:
        sim = np.dot(embed_fn(instruction), item["embedding"])

        for t in item.get("tasks", []):
            prm = t.get("params")

            if isinstance(prm, dict) and sim > best_score:
                best_score = sim
                best_params = prm

    return best_params


# ============================================
# 6. main推論
# ============================================

def generate_task_json_ver08(instruction, dataset, bm25_docs, df):

    # Step1: retrieval
    retrieved = retrieve_topk_hybrid(instruction, dataset, bm25_docs, df)

    # Step2: action
    action = decide_action(retrieved)

    # Step3: filter
    filtered = [r for r in retrieved if any(
        t.get("action") == action for t in r.get("tasks", [])
    )]

    # Step4: target
    target = extract_target(instruction)

    # Step5: params
    params = select_params(instruction, filtered)

    return {
        "tasks": [
            {
                "action": action,
                "target": target,
                "constraints": [],
                "params": params
            }
        ]
    }


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14370.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# ============================================
# 評価関数（ver08対応）
# ============================================

import json


# ============================================
# 背景意図:
# - 改善の方向性を明確にするため分解評価
# - totalだけだと原因が見えない
# ============================================


def safe_get(d, keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d


# ============================================
# action評価
# ============================================

def score_action(pred, gt):
    p = safe_get(pred, ["tasks", 0, "action"])
    g = safe_get(gt, ["tasks", 0, "action"])
    return int(p == g)


# ============================================
# target評価（部分一致OK）
# ============================================

def score_target(pred, gt):
    p = safe_get(pred, ["tasks", 0, "target"], "")
    g = safe_get(gt, ["tasks", 0, "target"], "")

    if not p or not g:
        return 0

    # 部分一致許容
    return int(p in g or g in p)


# ============================================
# params評価（キー一致 + 値一致）
# ============================================

def score_params(pred, gt):
    p = safe_get(pred, ["tasks", 0, "params"], {})
    g = safe_get(gt, ["tasks", 0, "params"], {})

    if not isinstance(p, dict) or not isinstance(g, dict):
        return 0

    if len(g) == 0:
        return 1

    match = 0

    for k in g:
        if k in p and p[k] == g[k]:
            match += 1

    return match / len(g)


# ============================================
# 総合スコア
# ============================================

def score_total(pred, gt):
    a = score_action(pred, gt)
    t = score_target(pred, gt)
    p = score_params(pred, gt)

    # 背景意図:
    # action最重要
    # params次
    # targetは補助

    return 0.5 * a + 0.3 * p + 0.2 * t


# ============================================
# 全体評価
# ============================================

def evaluate_all(results, gts):

    scores = {
        "action": [],
        "target": [],
        "params": [],
        "total": []
    }

    for r, gt in zip(results, gts):

        pred = r["pred"]

        scores["action"].append(score_action(pred, gt))
        scores["target"].append(score_target(pred, gt))
        scores["params"].append(score_params(pred, gt))
        scores["total"].append(score_total(pred, gt))

    # 平均
    summary = {k: sum(v)/len(v) for k, v in scores.items()}

    return summary

In [6]:
# ============================================
# 7. main（修正版）
# ============================================

from copy import deepcopy

dataset_path = "/workspace/data/annotations_gt_task_ver09.json"
output_path = "./prediction_results_ver09.json"


# ============================================
# JSON安全化関数
# ============================================

def clean_for_json(obj):
    if isinstance(obj, dict):
        return {
            k: clean_for_json(v)
            for k, v in obj.items()
            if k != "embedding"
        }
    elif isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


# ============================================
# データ読み込み（分離）
# ============================================

with open(dataset_path, "r") as f:
    dataset_raw = json.load(f)

# embedding用コピー
dataset = deepcopy(dataset_raw)

# embedding付与（内部専用）
for item in dataset:
    item["embedding"] = embed_fn(item["instruction"])

# BM25 index
bm25_docs, df = build_bm25_index(dataset)

results = []

progress = tqdm(dataset)

for idx, (item, gt_raw) in enumerate(zip(dataset, dataset_raw)):

    instruction = item["instruction"]
    gt = gt_raw  # ← ここが重要（embeddingなし）

    print("\n" + "="*80)
    print(f"[Sample {idx}]")
    print("Instruction:", instruction)

    # ====================================
    # Step1: Retrieval
    # ====================================
    retrieved = retrieve_topk_hybrid(instruction, dataset, bm25_docs, df)

    print("\n--- Step1: Retrieval Top5 ---")
    for i, r in enumerate(retrieved[:5]):
        print(f"{i}: {r['instruction']}")

    # ====================================
    # Step2: Action
    # ====================================
    pred_action = decide_action(retrieved)
    gt_action = gt["tasks"][0]["action"]

    print("\n--- Step2: Action ---")
    print("GT :", gt_action)
    print("Pred:", pred_action)

    # ====================================
    # Step3: Filter
    # ====================================
    filtered = [
        r for r in retrieved
        if any(t.get("action") == pred_action for t in r.get("tasks", []))
    ]

    print(f"\nFiltered count: {len(filtered)}")

    # ====================================
    # Step4: Target
    # ====================================
    pred_target = extract_target(instruction)
    gt_target = gt["tasks"][0].get("target", "")

    print("\n--- Step4: Target ---")
    print("GT :", gt_target)
    print("Pred:", pred_target)

    # ====================================
    # Step5: Params
    # ====================================
    pred_params = select_params(instruction, filtered)
    gt_params = gt["tasks"][0].get("params", {})

    print("\n--- Step5: Params ---")
    print("GT :", json.dumps(gt_params, indent=2, ensure_ascii=False))
    print("Pred:", json.dumps(pred_params, indent=2, ensure_ascii=False))

    # ====================================
    # Step6: Final JSON
    # ====================================
    pred = {
        "tasks": [
            {
                "action": pred_action,
                "target": pred_target,
                "constraints": [],
                "params": pred_params
            }
        ]
    }

    print("\n--- Step6: Final ---")
    print("GT :", json.dumps(clean_for_json(gt), indent=2, ensure_ascii=False))
    print("Pred:", json.dumps(clean_for_json(pred), indent=2, ensure_ascii=False))

    # ====================================
    # スコア
    # ====================================
    a = score_action(pred, gt)
    t = score_target(pred, gt)
    p = score_params(pred, gt)
    total = score_total(pred, gt)

    print("\n--- Score ---")
    print(f"action: {a}, target: {t}, params: {p:.3f}, total: {total:.3f}")

    progress.set_postfix({"total": total})

    # ====================================
    # 保存用（clean必須）
    # ====================================
    results.append({
        "instruction": instruction,

        "gt": clean_for_json(gt),
        "pred": clean_for_json(pred),

        "debug": {
            "retrieved_top5": [r["instruction"] for r in retrieved[:5]],
            "action": {
                "gt": gt_action,
                "pred": pred_action
            },
            "target": {
                "gt": gt_target,
                "pred": pred_target
            },
            "params": {
                "gt": clean_for_json(gt_params),
                "pred": clean_for_json(pred_params)
            }
        },

        "score": {
            "action": a,
            "target": t,
            "params": p,
            "total": total
        }
    })


# ====================================
# 保存
# ====================================
with open(output_path, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nSaved: {output_path}")


# ====================================
# 集計（raw GT使用）
# ====================================
summary = evaluate_all(results, dataset_raw)

print("\n===== Summary =====")
for k, v in summary.items():
    print(f"{k}: {v:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s, total=0.8]


[Sample 0]
Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.

--- Step1: Retrieval Top5 ---
0: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
1: Execute a smooth and gradual zoom-in transition starting from the current medium shot and ending on a tight close-up of the subject's face. The motion should be perfectly linear and devoid of any jitter or stuttering throughout the entire duration of the clip. Ensure that the subject remains in sharp focus and centered in the frame as the camera tightens. The graphic overlays, such as the 'SQUAWK' news bar and the 'LIVE NEW YORK' badge, must remain clear and integrated naturally without flickering or being awkwardly cropped.
2: Apply a smooth, gradual zoom-in effect centered on the speaker on the left throughout the entire video. Ensure t

  0%|          | 0/100 [00:00<?, ?it/s, total=0.8]


--- Step5: Params ---
GT : {
  "new_scene": {
    "type": "indoor",
    "style": "automotive_showroom",
    "lighting": "soft",
    "depth": "shallow",
    "objects": [
      "cars"
    ]
  }
}
Pred: {
  "new_scene": {
    "type": "indoor",
    "style": "automotive_showroom",
    "lighting": "soft",
    "depth": "shallow",
    "objects": [
      "cars"
    ]
  }
}

--- Step6: Final ---
GT : {
  "video_path": "DaUJkmBvTKM_2_0to150.mp4",
  "class": "Visual Effect Editing",
  "subclass": "Background Change",
  "instruction": "Replace the entire outdoor background behind the speaker with a sleek, modern automotive showroom featuring soft studio lighting and blurred cars in the distance. Ensure that the speaker's outline, including the fine edges of his head and shoulders, is cleanly masked with no edge flickering or artifacts across all frames. The existing lower-third text and the 'an' logo in the top right must remain perfectly legible and completely unaffected by the background modific

  0%|          | 0/100 [00:00<?, ?it/s, total=0.8]

...


--- Step5: Params ---
GT : {
  "style": "anime"
}
Pred: {
  "style": "anime"
}

--- Step6: Final ---
GT : {
  "video_path": "CMkYw4dp_NI_6_0to363.mp4",
  "class": "Style Editing",
  "subclass": "Anime",
  "instruction": "Transform the entire video into a high-quality Japanese anime style, characterized by sharp line art and vibrant, cel-shaded coloring. Ensure the man's facial expressions and features are accurately translated into the anime aesthetic while preserving his identity. The dark cookie should be stylized with a subtle hand-drawn texture, and the lighting should emphasize a bright, cinematic anime atmosphere. Throughout the clip, maintain temporal consistency to prevent flickering, ensuring clean edges and a professional finish.",
  "tasks": [
    {
      "action": "apply_style",
      "target": "full_frame",
      "constraints": [],
      "params": {
        "style": "anime"
      }
    },
    {
      "action": "preserve_identity",
      "target": "subjects",
      "constr

  0%|          | 0/100 [00:05<?, ?it/s, total=0.8]


--- Step1: Retrieval Top5 ---
0: Modify the video so that the two individuals in the car bring their coffee cups together to perform a toast. Ensure the motion of their arms and cups is fluid and realistic while maintaining their facial expressions and the interior details of the vehicle. The transition should be seamless with no flickering or distortions in the background throughout the entire sequence.
1: Modify the video so the man slowly turns his head to face the camera and offers a gentle smile. Maintain his original appearance and the details of the street background throughout the movement. Ensure the motion is fluid and realistic, with no flickering or artifacts.
2: Apply a smooth, gradual zoom-in effect to the entire video, focusing centrally on the two individuals at the table. Maintain the sharpness of the subjects and the detailed background props throughout the movement. Ensure the transition is steady with no flickering or jerky motions.
3: Modify the video so that the 

  0%|          | 0/100 [00:05<?, ?it/s, total=0.8]


--- Step5: Params ---
GT : {
  "style": "watercolor"
}
Pred: {
  "style": "watercolor"
}

--- Step6: Final ---
GT : {
  "video_path": "QrJFqor5GSs_35_0to216.mp4",
  "class": "Style Editing",
  "subclass": "Watercolor",
  "instruction": "Transform the video into a soft watercolor painting style, preserving the fluid motions of the makeup application.",
  "tasks": [
    {
      "action": "apply_style",
      "target": "full_frame",
      "constraints": [],
      "params": {
        "style": "watercolor"
      }
    }
  ]
}
Pred: {
  "tasks": [
    {
      "action": "apply_style",
      "target": "cat",
      "constraints": [],
      "params": {
        "style": "watercolor"
      }
    }
  ]
}

--- Score ---
action: 1, target: 0, params: 1.000, total: 0.800

[Sample 98]
Instruction: Replace the pink swirl frosting on top of the cupcake with a vibrant scoop of mint green frosting that has a creamy, textured surface. Maintain the original position and scale of the frosting so that the per